# Supply Chain Distribution Analysis

**Author:** Joseph Viscione  
**Tools:** Excel, Python, MySQL, Streamlit  
**Data Source:** DataCo Smart Supply Chain Dataset  

## Project Overview

This project analyzes a global supply chain dataset to uncover insights around product profitability, delivery performance, customer segmentation, and regional revenue trends. The full pipeline moves from raw data cleaning in Python to a normalized MySQL relational database, then to a Streamlit dashboard for business stakeholders.

## Business Questions

1. Which product categories generate the most revenue and profit?
2. Which markets and regions drive the most sales?
3. How does delivery performance vary by shipping mode?
4. Which products fall into ABC classification tiers A, B, and C?
5. Which customer segments are most valuable?
6. How does revenue trend month over month?

## Step 1 | Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Load raw data
df = pd.read_csv('data/DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Shape: (180519, 53)

Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Sta

## Step 2 | Data Inspection

In [10]:
# Preview first few rows
print(df.head())

# Check data types and null counts
print("\nData Types and Null Counts:")
print(df.info())

# Summary statistics for numeric columns
print("\nSummary Statistics:")
print(df.describe())

       type  days_for_shipping_real  days_for_shipment_scheduled  \
0     DEBIT                       3                            4   
1  TRANSFER                       5                            4   
2      CASH                       4                            4   
3     DEBIT                       3                            4   
4   PAYMENT                       2                            4   

   benefit_per_order  sales_per_customer   delivery_status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   late_delivery_risk  category_id   category_name customer_city  ...  \
0                   0           73  Sporting Goods        Caguas  ...   
1                   1           73  Sporting Goods        Caguas  ...   

## Step 3 | Data Cleaning

In [3]:
# Drop columns not needed for analysis
cols_to_drop = [
    'Customer Email', 'Customer Password', 'Customer Street',
    'Product Description', 'Product Image', 'Order Zipcode',
    'Latitude', 'Longitude'
]
df.drop(columns=cols_to_drop, inplace=True)

# Rename columns to clean snake_case
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# Convert date columns
df['order_date_dateorders'] = pd.to_datetime(df['order_date_dateorders'])
df['shipping_date_dateorders'] = pd.to_datetime(df['shipping_date_dateorders'])

# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Check nulls
print("Null counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f"\nShape after cleaning: {df.shape}")

Null counts:
customer_lname      8
customer_zipcode    3
dtype: int64

Shape after cleaning: (180519, 45)


In [4]:
# Fill minor nulls
df['customer_lname'] = df['customer_lname'].fillna('Unknown')
df['customer_zipcode'] = df['customer_zipcode'].fillna(0)

# Confirm no nulls remain
print("Remaining nulls:", df.isnull().sum().sum())

Remaining nulls: 0


## Step 4 | Build Normalized Tables for MySQL

In [5]:
# Customers table
customers = df[[
    'customer_id', 'customer_fname', 'customer_lname',
    'customer_segment', 'customer_city', 'customer_country',
    'customer_state', 'customer_zipcode'
]].drop_duplicates(subset='customer_id').reset_index(drop=True)

# Categories table
categories = df[[
    'category_id', 'category_name', 'department_id', 'department_name'
]].drop_duplicates(subset='category_id').reset_index(drop=True)

# Products table
products = df[[
    'product_card_id', 'product_name', 'product_price',
    'product_status', 'category_id'
]].drop_duplicates(subset='product_card_id').reset_index(drop=True)

# Orders table
orders = df[[
    'order_id', 'order_customer_id', 'order_date_dateorders',
    'order_status', 'shipping_mode', 'market',
    'order_region', 'order_city', 'order_country'
]].drop_duplicates(subset='order_id').reset_index(drop=True)

# Order items table
order_items = df[[
    'order_item_id', 'order_id', 'product_card_id',
    'order_item_quantity', 'sales', 'order_item_discount_rate',
    'order_item_profit_ratio', 'order_profit_per_order',
    'days_for_shipping_real', 'days_for_shipment_scheduled',
    'delivery_status', 'late_delivery_risk'
]].reset_index(drop=True)

print(f"Customers: {customers.shape}")
print(f"Categories: {categories.shape}")
print(f"Products: {products.shape}")
print(f"Orders: {orders.shape}")
print(f"Order Items: {order_items.shape}")

Customers: (20652, 8)
Categories: (51, 4)
Products: (118, 5)
Orders: (65752, 9)
Order Items: (180519, 12)


## Step 5 | Load Tables into MySQL

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    database="supply_chain"
)

engine = create_engine(connection_url)

# Load tables
customers.to_sql('customers', con=engine, if_exists='replace', index=False)
categories.to_sql('categories', con=engine, if_exists='replace', index=False)
products.to_sql('products', con=engine, if_exists='replace', index=False)
orders.to_sql('orders', con=engine, if_exists='replace', index=False)
order_items.to_sql('order_items', con=engine, if_exists='replace', index=False)

print("All tables loaded successfully.")

All tables loaded successfully.


## Step 6 | ABC Classification

In [7]:
# Pull product revenue data from MySQL
query = """
    SELECT 
        p.product_name,
        c.category_name,
        SUM(oi.sales) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_card_id = p.product_card_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY p.product_name, c.category_name
    ORDER BY total_revenue DESC
"""

product_revenue = pd.read_sql(query, con=engine)

# Calculate cumulative revenue percentage
product_revenue['cumulative_revenue'] = product_revenue['total_revenue'].cumsum()
product_revenue['cumulative_pct'] = product_revenue['cumulative_revenue'] / product_revenue['total_revenue'].sum() * 100

# Assign ABC tiers
def abc_class(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

product_revenue['abc_class'] = product_revenue['cumulative_pct'].apply(abc_class)

print(product_revenue['abc_class'].value_counts())
print(product_revenue.head(10))

abc_class
C    95
B    16
A     7
Name: count, dtype: int64
                                    product_name         category_name  \
0      Field & Stream Sportsman 16 Gun Fire Safe               Fishing   
1               Perfect Fitness Perfect Rip Deck                Cleats   
2  Diamondback Women's Serene Classic Comfort Bi      Camping & Hiking   
3              Nike Men's Free 5.0+ Running Shoe      Cardio Equipment   
4           Nike Men's Dri-FIT Victory Golf Polo       Women's Apparel   
5                    Pelican Sunstream 100 Kayak          Water Sports   
6        Nike Men's CJ Elite 2 TD Football Cleat        Men's Footwear   
7               O'Brien Men's Neoprene Life Vest  Indoor/Outdoor Games   
8  Under Armour Girls' Toddler Spine Surge Runni         Shop By Sport   
9                                    Dell Laptop             Computers   

   total_revenue  cumulative_revenue  cumulative_pct abc_class  
0   6.929654e+06        6.929654e+06       18.838395        

## Step 7 | Export CSVs for Visualization

In [8]:
import os
os.makedirs('outputs', exist_ok=True)

# Export ABC classification
product_revenue.to_csv('outputs/abc_classification.csv', index=False)

# Export query results for Tableau
revenue_by_category = pd.read_sql("""
    SELECT c.category_name, COUNT(oi.order_item_id) AS total_orders,
    SUM(oi.sales) AS total_revenue, SUM(oi.order_profit_per_order) AS total_profit,
    ROUND(SUM(oi.order_profit_per_order) / SUM(oi.sales) * 100, 2) AS profit_margin_pct
    FROM order_items oi
    JOIN products p ON oi.product_card_id = p.product_card_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category_name ORDER BY total_revenue DESC
""", con=engine)
revenue_by_category.to_csv('outputs/revenue_by_category.csv', index=False)

revenue_by_segment = pd.read_sql("""
    SELECT c.customer_segment, COUNT(DISTINCT c.customer_id) AS total_customers,
    SUM(oi.sales) AS total_revenue, SUM(oi.order_profit_per_order) AS total_profit
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN customers c ON o.order_customer_id = c.customer_id
    GROUP BY c.customer_segment ORDER BY total_revenue DESC
""", con=engine)
revenue_by_segment.to_csv('outputs/revenue_by_segment.csv', index=False)

shipping_performance = pd.read_sql("""
    SELECT o.shipping_mode, COUNT(oi.order_item_id) AS total_orders,
    SUM(oi.sales) AS total_revenue,
    ROUND(AVG(oi.days_for_shipping_real), 2) AS avg_actual_days,
    ROUND(SUM(oi.late_delivery_risk) / COUNT(oi.order_item_id) * 100, 2) AS late_delivery_pct
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY o.shipping_mode ORDER BY total_revenue DESC
""", con=engine)
shipping_performance.to_csv('outputs/shipping_performance.csv', index=False)

monthly_trend = pd.read_sql("""
    SELECT YEAR(o.order_date_dateorders) AS order_year,
    MONTH(o.order_date_dateorders) AS order_month,
    SUM(oi.sales) AS total_revenue, SUM(oi.order_profit_per_order) AS total_profit
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY order_year, order_month ORDER BY order_year, order_month
""", con=engine)
monthly_trend.to_csv('outputs/monthly_trend.csv', index=False)

print("All CSVs exported successfully.")

All CSVs exported successfully.


In [9]:
# KPI Summary
kpi_summary = pd.DataFrame([{
    'total_revenue': round(df['sales'].sum(), 2),
    'total_orders': len(df),
    'avg_profit_margin': round((df['order_profit_per_order'].sum() / df['sales'].sum()) * 100, 2),
    'late_delivery_rate': round((df['late_delivery_risk'].sum() / len(df)) * 100, 2)
}])
kpi_summary.to_csv('outputs/kpi_summary.csv', index=False)

# ABC Summary — count of products per class
abc_summary = product_revenue.groupby('abc_class').size().reset_index(name='product_count')
abc_summary.to_csv('outputs/abc_summary.csv', index=False)

# Fix monthly trend — add proper date column
monthly_trend['date'] = pd.to_datetime(monthly_trend['order_year'].astype(str) + '-' + monthly_trend['order_month'].astype(str) + '-01')
monthly_trend.to_csv('outputs/monthly_trend.csv', index=False)

# Revenue by region
revenue_by_region = pd.read_sql("""
    SELECT 
        o.market,
        o.order_region,
        COUNT(oi.order_item_id) AS total_orders,
        SUM(oi.sales) AS total_revenue,
        SUM(oi.order_profit_per_order) AS total_profit
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY o.market, o.order_region
    ORDER BY total_revenue DESC
""", con=engine)
revenue_by_region.to_csv('outputs/revenue_by_region.csv', index=False)

# Top 10 categories only
revenue_by_category_top10 = revenue_by_category.head(10)
revenue_by_category_top10.to_csv('outputs/revenue_by_category_top10.csv', index=False)

print("All additional CSVs exported successfully.")
print(f"\nKPI Summary:\n{kpi_summary}")
print(f"\nABC Summary:\n{abc_summary}")

All additional CSVs exported successfully.

KPI Summary:
   total_revenue  total_orders  avg_profit_margin  late_delivery_rate
0    36784735.01        180519              10.78               54.83

ABC Summary:
  abc_class  product_count
0         A              7
1         B             16
2         C             95


## Step 8: Verify Outputs for Streamlit

In [11]:
import os

expected_files = [
    'outputs/kpi_summary.csv',
    'outputs/revenue_by_category_top10.csv',
    'outputs/revenue_by_segment.csv',
    'outputs/abc_summary.csv',
    'outputs/monthly_trend.csv',
    'outputs/shipping_performance.csv'
]

for f in expected_files:
    status = "Found" if os.path.exists(f) else "MISSING"
    print(f"{status}: {f}")

Found: outputs/kpi_summary.csv
Found: outputs/revenue_by_category_top10.csv
Found: outputs/revenue_by_segment.csv
Found: outputs/abc_summary.csv
Found: outputs/monthly_trend.csv
Found: outputs/shipping_performance.csv


## Step 9: Data Validation

In [13]:
import pandas as pd

files = {
    'kpi_summary': 'outputs/kpi_summary.csv',
    'revenue_by_category_top10': 'outputs/revenue_by_category_top10.csv',
    'revenue_by_segment': 'outputs/revenue_by_segment.csv',
    'abc_summary': 'outputs/abc_summary.csv',
    'monthly_trend': 'outputs/monthly_trend.csv',
    'shipping_performance': 'outputs/shipping_performance.csv'
}

for name, path in files.items():
    df = pd.read_csv(path)
    nulls = df.isnull().sum().sum()
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols, {nulls} nulls")
    print(f"  Columns: {df.columns.tolist()}\n")

kpi_summary: 1 rows, 4 cols, 0 nulls
  Columns: ['total_revenue', 'total_orders', 'avg_profit_margin', 'late_delivery_rate']

revenue_by_category_top10: 10 rows, 5 cols, 0 nulls
  Columns: ['category_name', 'total_orders', 'total_revenue', 'total_profit', 'profit_margin_pct']

revenue_by_segment: 3 rows, 4 cols, 0 nulls
  Columns: ['customer_segment', 'total_customers', 'total_revenue', 'total_profit']

abc_summary: 3 rows, 2 cols, 0 nulls
  Columns: ['abc_class', 'product_count']

monthly_trend: 37 rows, 5 cols, 0 nulls
  Columns: ['order_year', 'order_month', 'total_revenue', 'total_profit', 'date']

shipping_performance: 4 rows, 5 cols, 0 nulls
  Columns: ['shipping_mode', 'total_orders', 'total_revenue', 'avg_actual_days', 'late_delivery_pct']



## Step 10: Handoff to Streamlit

The `outputs/` folder contains all CSVs required by the Streamlit dashboard. Here is how the full pipeline flows:

1. Raw data is loaded and cleaned in Python
2. Normalized tables are pushed to MySQL
3. SQL queries aggregate the data into summary tables
4. Results are exported as CSVs to `outputs/`
5. Streamlit reads those CSVs directly to power the dashboard

To run the dashboard locally: `streamlit run app.py`